# Reproducing Komori et al. (2026) from DANDI:001425: decoding of spontaneous pupil dilation

**Paper.** Komori Y, Mizusaki K, Tomita K, Yoneda T, Matoba K, Morita T (2026). Mesoscopic cortical
activities associated with pupil-linked perceptions inferred via explainable machine learning.
bioRxiv, [doi:10.64898/2026.05.27.726399](https://doi.org/10.64898/2026.05.27.726399). The claims
file (`claims/komori-2026.yaml`) was written from the preprint posted June 1, 2026, with its
Methods, legends and Supplementary Figures 3 to 6.

**Dandiset.** [DANDI:001425](https://dandiarchive.org/dandiset/001425/0.250705.0947), version
0.250705.0947 ("BraiDyn-BC: mouse cued-lever-pull task dataset", Kondo et al. 2025). The paper
cites the dandiset without a version. Two versions are published, 0.250514.0602 (1814 assets) and
0.250705.0947 (1838 assets); the most recent is pinned, as the protocol asks when no version is
cited. The paper was posted in 2026, after both, so either could be the one it read. The data check
shows that 6 of the 9 files read here are the same assets in both versions; the other 3 were
replaced in the later version, and the earlier ones were not compared.
The paper uses only the resting-state recordings of this dandiset; its own recordings
(somatosensory and visual stimulation) are not deposited.

**What is tested.** The six resting-state claims. The paper's pipeline is rebuilt from its
Methods: hemodynamic correction, SVD and JADE ICA of the cortical images, pupil labels around the
dilation peak of each 40 s epoch, a GRU decoder with five-fold cross-validation and null models
trained on labels shuffled within epochs. The open details R1 to R10 of the claims file's opening
comment are followed as it names them; where several readings are named, the notebook runs the
main one and as many others as time allows, and says which were not run.

This notebook tests the three decoding claims; the companion notebook
`reproduce_komori_2026_features.ipynb` tests the IC count, the window, the VIF and the SHAP claims.

| Claim | Notebook | Status |
|---|---|---|
| rest-decoding-above-null | decoding | tested (R4 b and ICA restarts in the features notebook) |
| rest-loss-accuracy-vs-null | decoding | tested |
| rest-needs-0-1-hz | decoding | tested |
| rest-ic-count-and-window | features | tested |
| rest-vif-below-10 | features | tested |
| rest-shap-medial-anterior-first | features | tested |
| fig1-*, fig2-*, fig3-*, fig4-shap-*, fig4-mapped-shap-regions, supp2-*, supp4-*, supp8-*, supp10-* (18 claims) | none | blocked: the stimulation recordings are not deposited |

In [ ]:
import pupilwf as pw                       # first: it pins the numerical kernels (pw.KERNELS)

import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from threadpoolctl import threadpool_info

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
print("kernels:", pw.KERNELS, "; linear algebra:", sorted({(i['internal_api'], i.get('architecture')) for i in threadpool_info()}, key=str))

started = time.time()
assets = pw.list_assets()
rows = pw.resting_table(assets)
chosen = pw.choose_sessions(rows)          # R1 (a): three mice, first three resting sessions each
chosen_rows = [r for mouse in chosen for r in mouse]
sessions = pw.load_sessions(chosen_rows)
print(f"streamed and reduced {len(sessions)} sessions from DANDI:{pw.DANDISET} version {pw.VERSION} "
      f"in {time.time() - started:.0f} s")

In [ ]:
t0 = time.time()
D = pw.decompose(sessions, hemo="fit_hp_subtract_raw")       # R2 (a), main reading
comp = pw.compactness(D["maps"], D["mask"])
# features for VIF-based selection: every frame of the 13 epochs, 0.1 Hz low-pass (R6)
def vif_frames(ics, cutoff=0.1):
    out = []
    for s, ic in zip(sessions, ics):
        a, b = pw.epochs_of(s["t"])[0][0], pw.epochs_of(s["t"])[-1][1]
        out.append(pw.normalise_features(ic, cutoff)[a:b])
    return np.vstack(out)
F01 = vif_frames(D["ics"])
keep10, removed10 = pw.select_ics(F01, comp, 10, "kept")                  # VIF of the paper's code (uncentred)
keep10c, removed10c = pw.select_ics(F01, comp, 10, "kept", centred=True)  # variant: VIF with an intercept
print(f"decomposition in {time.time() - t0:.0f} s; JADE converged in {D['sweeps']} sweeps; kept ICs ({len(keep10)}) {[int(k) for k in keep10]}; "
      f"with the centred VIF ({len(keep10c)}) {[int(k) for k in keep10c]}")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
tab = pd.DataFrame(rows)
print("Resting-state NWB files in the dandiset:", len(tab), "from", tab.subject.nunique(), "mice")
print("Files per mouse:", tab.groupby("subject").size().value_counts().sort_index().to_dict(),
      "(number of files: number of mice)")
print("Files without behaviour (widefield only, no pupil):", int((~tab.behavior).sum()))
print(tab[~tab.behavior][["subject", "day"]].to_string(index=False))
print("Paper, Methods, Resting state: 25 mice, '2-3 experiments per animal'.")
print("\nChosen (R1 a): ", [(r["subject"], r["day"], r["date"]) for r in chosen_rows])
old = {a["path"]: a["asset_id"] for a in pw.list_assets("0.250514.0602")}
print("Chosen files with the same asset (same content) in version 0.250514.0602:",
      sum(old.get(r["path"]) == r["asset_id"] for r in chosen_rows), "of", len(chosen_rows))

In [ ]:
info = pd.DataFrame([{
    "subject": s["subject"], "day": s["day"], "frames": s["n_frames"],
    "rate_hz": round(1 / np.median(np.diff(s["t"])), 3), "duration_s": round(s["t"][-1], 1),
    "uv_after_blue_ms": round(1000 * s["t_uv_offset"], 1), "pupil_on_img_clock": s["pupil_t_equal"],
    "dff_on_img_clock": s["dff_t_equal"], "pupil_nan": s["pupil_nan"],
    "pupil_px_max_deposited": float(f"{np.nanmax(s['pupil_filled']):.3g}"), "pupil_replaced": s["pupil_replaced"],
    "pupil_px_min": round(float(s["pupil"].min()), 1), "pupil_px_median": round(float(np.median(s["pupil"])), 1),
    "pupil_px_max": round(float(s["pupil"].max()), 1),
    "kp_width_bad": s["pupil_kp_bad"], "r_diam_kp": round(float(np.corrcoef(s["pupil"], s["pupil_kp"])[0, 1]), 3),
    "cortex_px": s["n_pixels"], "svd34_var": round(s["svd_var_explained"], 4),
    "raw_blue": round(s["raw_mean"][0]), "raw_uv": round(s["raw_mean"][1]),
    "read_s": round(s["read_s"])} for s in sessions])
print(info.to_string(index=False))
print("\nPaper (Methods, Resting state): 60 Hz acquisition alternating 405/470 nm, 36,000 frames per 600 s "
      "session, i.e. 18,000 frames and 30 Hz per channel; frames 288 x 288 after down-sampling.")
print("Frames per channel:", sorted(set(info.frames)), "; frame shape in the files: 288 x 288 (binned 2x2 here)")
print("Hemodynamic regression slope of F470 on F405 (34 components per session): median",
      round(float(np.median(np.concatenate([s["coef"][:, 0] for s in sessions]))), 3),
      "range", np.round(np.percentile(np.concatenate([s["coef"][:, 0] for s in sessions]), [0, 100]), 2))
slopes = np.concatenate([s["coef"][:, 0] for s in sessions])
print("Slopes: percentiles 5, 25, 50, 75, 95:", np.round(np.percentile(slopes, [5, 25, 50, 75, 95]), 2).tolist(),
      "; negative:", int((slopes < 0).sum()), "of", len(slopes))
print("Pupil columns above: the deposited maximum, the frames replaced as failed detections (helper, clean_pupil), and "
      "the range after replacement. r_diam_kp: correlation of the cleaned diameter with the keypoint width (R3 b).")
for r in chosen_rows[:2]:
    print("Channel check,", r["subject"], r["day"], pw.channel_check(r))
f = pw.open_nwb(chosen_rows[0]["asset_id"], block_mb=2)
for plane in ("ImagingPlane_blue", "ImagingPlane_UV"):
    g = f["general/optophysiology/" + plane]
    print(plane, "excitation_lambda", g["excitation_lambda"][()], "imaging_rate", g["imaging_rate"][()],
          "indicator", g["indicator"][()].decode(), "grid_spacing", g["grid_spacing"][:])
print("acquisition/widefield_blue description:", f["acquisition/widefield_blue"].attrs["description"])
print("dFF_B description:", f["processing/ophys/DfOverF/dFF_B"].attrs["description"])
print("pupil diameter unit:", f["processing/behavior/pupil_tracking/diameter/data"].attrs["unit"])
print("genotype:", f["general/subject/genotype"][()].decode())
f.close()

Every session holds 18,000 frames per channel at 29.95 Hz over 600 s, as the paper describes
(36,000 frames at 60 Hz, alternating). The pupil diameter and the deposited dF/F are on the
imaging timestamps exactly. The deposited pupil diameter has failed detections that the files do
not flag: in VG1-GC#51 day0 and VG1-GC#53 day1 its maximum is 2.88e21 and 1.05e11 pixels, and
148 and 57 frames there are above three times the median (besides 768 and 269 missing values);
these are replaced before any label is made. The channel check shows the calcium signal in
`widefield_blue` (dF/F SD 0.050 and 0.028 against 0.012 and 0.014 in `widefield_UV`), as its
description says, but the imaging-plane metadata give 405 nm for the blue plane and 470 nm for the
UV plane: the excitation wavelengths are swapped in the metadata.

In [ ]:
# Recordings deposited twice: identical pupil traces or identical mean fluorescence would show it.
P = np.array([s["pupil"][:18000] for s in sessions])
C = np.corrcoef(P)
np.fill_diagonal(C, np.nan)
print("Largest pupil-trace correlation between two different sessions:", round(float(np.nanmax(C)), 3))
means = [s["raw_mean"] for s in sessions]
print("Distinct (blue, UV) mean raw intensities:", len(set(means)), "of", len(means))

In [ ]:
# Clock check. The pupil and the imaging share the timestamps (above). Whether the pupil trace is
# aligned with the imaging in fact is checked with the cross-correlation between the pupil and the
# mean cortical activity (deposited hemodynamics-corrected dF/F, averaged over the 44 ROIs): arousal
# links the two, and a misaligned trace would move the peak. Control: the same with the pupil
# circularly shifted by 100 to 500 s.
lags = np.arange(-300, 301, 3)
rows_clk = []
for s in sessions:
    p = pw.butter(s["pupil"], 0.2)
    p = (p - p.mean()) / p.std()
    g = pw.butter(np.nanmean(s["dff_deposited"], axis=1), 0.2)
    g = (g - g.mean()) / g.std()
    cc = np.array([np.mean(p[max(0, -l):len(p) - max(0, l)] * g[max(0, l):len(g) - max(0, -l)]) for l in lags])
    sh = []
    for k in range(5):
        q = np.roll(p, 3000 + 3000 * k)
        sh.append(np.max(np.abs([np.mean(q[max(0, -l):len(q) - max(0, l)] * g[max(0, l):len(g) - max(0, -l)]) for l in lags[::10]])))
    i = int(np.argmax(np.abs(cc)))
    rows_clk.append(dict(subject=s["subject"], day=s["day"], peak_r=round(float(cc[i]), 3),
                         lag_s=round(float(lags[i] / pw.FS), 2), r_at_0=round(float(cc[lags == 0][0]), 3),
                         shifted_max_abs_r=round(float(np.max(sh)), 3)))
print("Positive lag: cortical activity follows the pupil.")
print(pd.DataFrame(rows_clk).to_string(index=False))

The pupil and the mean cortical activity correlate in every session, with the pupil lagging
cortical activity by 1.1 to 2.3 s in seven of nine sessions (5.4 and 7.6 s in VG1-GC#52 day7 and
day1). The peak correlation exceeds the largest one with the pupil circularly shifted in all nine
sessions, narrowly in VG1-GC#52 day1 (0.210 against 0.206) and VG1-GC#53 day1 (0.187 against
0.169). This is consistent with the pupil and the imaging sharing a clock; it does not prove it,
and the clock of those two sessions is the least supported.

In [ ]:
print("Common cortical grid points (atlas grid, step", pw.GRID_STEP, "atlas pixels):", int(D["mask"].sum()))
print("SVD components available:", D["n_svd"], "(paper: 400; the corrected data of each recording have rank 34)")
print("Cumulative variance at 10, 30, 200 components:", np.round(D["cum_var"][[9, 29, min(199, D['n_svd'] - 1)]], 4),
      "(paper: > 99.5% at 200)")
print("Frames used to estimate the decomposition (every fifth 40 s epoch):", D["n_sel_frames"])
print("IC compactness (share of squared weight in the largest supra-half-maximum region):")
print(pd.Series(comp).round(3).to_string())
print("VIF of all 30 ICs (0.1 Hz features): max", round(float(pw.vif(F01).max()), 2),
      "uncentred (no intercept, the paper's code);", round(float(pw.vif(F01, centred=True).max()), 2), "centred")
print("Kept for 10 ICs (reading a), uncentred VIF:", [int(k) for k in keep10], "removed for VIF:", [int(k) for k in removed10],
      "max VIF of the kept:", round(float(pw.vif(F01[:, keep10]).max()), 2))
print("Kept for 10 ICs (reading a), centred VIF:", [int(k) for k in keep10c], "removed for VIF:", [int(k) for k in removed10c],
      "max VIF of the kept:", round(float(pw.vif(F01[:, keep10c], centred=True).max()), 2))
roi = sessions[0]["roi_atlas"]
names = sessions[0]["roi_names"]
fig, axes = plt.subplots(2, 5, figsize=(13, 5.5))
for ax, j in zip(axes.flat, keep10):
    m = D["maps"][j]
    lim = np.nanmax(np.abs(m))
    ax.imshow(m, cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.contour(roi.any(axis=0), [0.5], colors="k", linewidths=0.5)
    ax.set_title(f"IC {j}  comp {comp[j]:.2f}", fontsize=9)
    ax.axis("off")
plt.suptitle(f"Spatial maps of the {len(keep10)} kept ICs on the common atlas grid (colour limits: +/- max |weight|)")
plt.tight_layout()
plt.show()

In [ ]:
ds_main = pw.make_dataset(sessions, D["ics"], keep10, cutoff=0.1)
lab = pd.DataFrame(ds_main["info"])
print("Epochs per recording:", lab.groupby("rec").size().unique(), "; samples per epoch window:", sorted(lab.n.unique()))
print("Positive frames per epoch window (of", lab.n.max(), "), by recording:")
print(lab.groupby(["subject", "day"]).n_pos.describe()[["mean", "min", "50%", "max"]].round(1).to_string())
print("Epochs whose window has no positive frame:", int((lab.n_pos == 0).sum()), "of", len(lab),
      "; all frames positive:", int((lab.n_pos == lab.n).sum()))
print("Samples:", ds_main["X"].shape, "positive share", round(float(ds_main["y"].mean()), 3))
print("Feature range:", float(ds_main["X"].min()), float(ds_main["X"].max()))
lab5 = pd.DataFrame(pw.make_dataset(sessions, D["ics"], keep10, 0.1, selection="rule")["info"])
print("Epochs kept by the selection rule R5 (b):", int(lab5.keep.sum()), "of", len(lab5))
print("Epochs whose pupil peak is at the epoch's first frame or within 0.5 s of its end (the label window then",
      "reaches into the next epoch):", int(lab.edge.sum()), "of", len(lab), "; among the epochs kept by R5 (b):",
      int((lab.edge & lab5.keep).sum()))

In [ ]:
# Hemodynamic correction: the ROI-averaged corrected dF/F of this attempt against the deposited ROI dF/F,
# which the dataset's authors corrected themselves (both low-pass 0.1 Hz). Uncorrected 470 nm dF/F as reference.
hc = []
for s in sessions:
    h = pw.hemo_check(s)
    hc.append(dict(subject=s["subject"], day=s["day"], rois=len(h["corrected"]),
                   r_corrected_median=np.median(h["corrected"]), r_corrected_min=h["corrected"].min(),
                   r_uncorrected_median=np.median(h["uncorrected"]),
                   neg_slopes=int((s["coef"][:, 0] < 0).sum())))
print(pd.DataFrame(hc).round(3).to_string(index=False))

In [ ]:
# The two sessions with failed detections: deposited diameter, cleaned diameter, normalised pupil and labels.
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for ax, (subj, day) in zip(axes, (("VG1-GC#51", "day0"), ("VG1-GC#53", "day1"))):
    s = next(x for x in sessions if x["subject"] == subj and x["day"] == day)
    pn = pw.pupil_norm(s["pupil"])
    labs = pw.epoch_labels(s["t"], pn)
    ax.plot(s["t"], pn, lw=0.6, color="k", label="normalised pupil (cleaned, 0.2 Hz)")
    ax.axhline(pw.THRESH, color="r", lw=0.6)
    for L in labs:
        ax.axvspan(s["t"][L["frames"][0]], s["t"][L["frames"][-1]], color="g" if L["y"].any() else "0.8", alpha=0.4)
    dep = s["pupil_filled"]
    print(f"{subj} {day}: deposited diameter median {np.median(dep):.1f}, frames above 3x median {int((dep > 3 * np.median(dep)).sum())}, "
          f"cleaned max {s['pupil'].max():.1f} ({s['pupil'].max() / np.median(s['pupil']):.2f} x median); "
          f"normalised pupil above 0.6 in {100 * float((pn > pw.THRESH).mean()):.1f}% of frames; "
          f"epoch windows with a positive label {sum(L['y'].any() for L in labs)} of {len(labs)}; "
          f"correlation with the keypoint width {np.corrcoef(s['pupil'], s['pupil_kp'])[0, 1]:.3f}")
    ax.set_title(f"{subj} {day}: label windows shaded (green: holds a positive label); red: threshold 0.6", fontsize=9)
    ax.set_ylim(0, 1)
axes[1].set_xlabel("time (s)")
plt.tight_layout()
plt.show()

With the whole recording's minimum and maximum, the labels are mostly constant within an epoch
window: 42 of 117 windows hold no positive frame and 37 are positive throughout, and VG1-GC#53 day1
has none at all (its cleaned diameter still reaches 2.88 times its median, and its normalised
pupil is above 0.6 in 0.7% of frames). The selection rule R5 (b) keeps 17 of 117 epochs. In 15 of
117 epochs (2 of the 17 kept by R5 b) the peak is at the epoch's first frame or within 0.5 s of its
end, so the label window can reach into the next epoch, which may sit in another fold.

The hemodynamic slopes of F470 on F405 are negative in 54 of the 306 components (5th percentile
-0.63), where the correction adds the 405 nm signal. The corrected ROI signal correlates with the
dataset's own corrected ROI dF/F with a median r of 0.26 to 0.70 per session, against 0.13 to 0.61
for the uncorrected signal; the correction brings it closer in six sessions and further away in
three (VG1-GC#51 day7, VG1-GC#52 day0, VG1-GC#53 day0). Agreement is moderate in all of them.

## rest-decoding-above-null

> A GRU decoder of independent components of resting cortical activity, low-pass filtered at 0.1 Hz,
> predicts spontaneous pupil dilation with a validation ROC-AUC significantly higher than that of
> null models trained on within-session shuffled labels. (Expected: model 0.82 +/- 0.082, null
> 0.61 +/- 0.061; paired t-test p < 0.05.)

Main reading combination: R1 a (3 mice x 3 sessions), R2 a (paper's correction on the frames,
coefficients applied to the measured signals), R3 a (deposited diameter), R4 a (samples in the
window), R6 (ICs selected for a count of 10, reading a, with the VIF of the paper's code; 7 pass), R7 a (folds of epochs), R8 a
(training labels shuffled), R9, at the claim's seed, under both readings of the epoch selection:
R5 a (none) and R5 b (the stimulus data's rule). The paper says the early- or never-crossing
sessions "were therefore excluded from all datasets" (Results, somatosensory, paragraph 1) and
that "Reliable prediction was achieved after excluding sessions..." (Discussion, Summary of
Results), so R5 b is the reading the paper best supports; the claims file says the paper states
no rule for resting data, and this is asked of a reader (questions/komori-2026.yaml). Each is run
under the claim's seed and the seed plus 1000 to 4000. Then the other readings one at a time
under R5 a, and every combination of the cheap readings under R5 b.

In [ ]:
SEED1 = pw.seed_of("rest-decoding-above-null")
t0 = time.time()
ds_rule = pw.make_dataset(sessions, D["ics"], keep10, cutoff=0.1, selection="rule")
cv_main, models_main = pw.run_cv(ds_main, SEED1, keep_models=True)
cv_rule = pw.run_cv(ds_rule, SEED1)
cols = ["fold", "n_train", "n_val", "pos_val", "model_epoch", "model_train_auc", "model_val_auc", "null_val_auc",
        "model_val_loss", "null_val_loss", "model_val_acc", "null_val_acc", "null_changed"]
for name, cv in (("R5 a (no epoch selection)", cv_main), ("R5 b (epoch selection rule)", cv_rule)):
    print(name)
    print(cv[cols].round(3).to_string(index=False))
    r = pw.paired(cv, "val_auc")
    print(f"validation AUC: model {r['model']:.3f}, null {r['null']:.3f}, paired t = {r['t']:.2f}, p = {r['p']:.3g}  "
          f"(expected 0.82 +/- 0.082 and 0.61 +/- 0.061)")
    print(f"model AUC - null AUC per fold: {np.round(cv.model_val_auc - cv.null_val_auc, 3).tolist()}")
    print(f"training AUC of the models: mean {cv.model_train_auc.mean():.3f}; share of training labels the "
          f"within-epoch shuffle changes: {cv.null_changed.mean():.3f}\n")
print(f"({time.time() - t0:.0f} s)")

In [ ]:
# Training curves, for comparison with Supplementary Figure 5B-C (mean over the five folds, per training epoch).
for name, cv in (("R5 a", cv_main), ("R5 b", cv_rule)):
    rows_h = []
    for kind in ("model", "null"):
        h = pd.DataFrame([dict(fold=f, **e) for f, hist in enumerate(cv[kind + "_hist"]) for e in hist])
        m = h.groupby("epoch")[["train_auc", "val_auc", "val_loss", "val_acc"]].mean()
        m.columns = [kind + "_" + c for c in m.columns]
        rows_h.append(m)
    print(name, "(epoch of lowest validation loss per fold, model:", cv.model_epoch.tolist(), "null:", cv.null_epoch.tolist(), ")")
    print(pd.concat(rows_h, axis=1).round(3).to_string())

In [ ]:
bym = pw.auc_by_mouse(ds_main, models_main)
print("Validation AUC by mouse (mean over folds):")
print(bym.groupby("mouse")[["model_auc", "null_auc"]].mean().round(3).to_string())

In [ ]:
seed_rows = []
cv_by_seed = {("R5 a", 0): cv_main, ("R5 b", 0): cv_rule}
for off in (1000, 2000, 3000, 4000):
    cv_by_seed[("R5 a", off)] = pw.run_cv(ds_main, SEED1 + off)
    cv_by_seed[("R5 b", off)] = pw.run_cv(ds_rule, SEED1 + off)
for (combo, off), cv in sorted(cv_by_seed.items()):
    r = pw.paired(cv, "val_auc")
    seed_rows.append(dict(combo=combo, seed=f"claim+{off}", model_auc=r["model"], null_auc=r["null"], t=r["t"], p=r["p"],
                          above=bool(r["p"] < 0.05 and r["model"] > r["null"]),
                          within_tol=bool(abs(r["model"] - 0.82) <= 0.082),
                          model_auc_per_fold=np.round(cv.model_val_auc, 3).tolist(),
                          null_auc_per_fold=np.round(cv.null_val_auc, 3).tolist()))
print(pd.DataFrame(seed_rows).round(4).to_string(index=False))

In [ ]:
# Other readings, one at a time under R5 a, at the claim's seed.
def variant(name, sess=sessions, ics=None, keep=None, ds_kw=None, cv_kw=None):
    t0 = time.time()
    ds = pw.make_dataset(sess, ics if ics is not None else D["ics"], keep if keep is not None else keep10,
                         **({"cutoff": 0.1} | (ds_kw or {})))
    cv = pw.run_cv(ds, SEED1, **(cv_kw or {}))
    out = {"reading": name, "samples": len(ds["y"]), "pos": round(float(ds["y"].mean()), 3)}
    for m in ("val_auc", "val_loss", "val_acc"):
        r = pw.paired(cv, m)
        out[m + "_model"], out[m + "_null"], out[m + "_p"] = r["model"], r["null"], r["p"]
    out["s"] = round(time.time() - t0)
    return out, cv

D_hp = pw.decompose(sessions, hemo="hp_corrected")
comp_hp = pw.compactness(D_hp["maps"], D_hp["mask"])
keep_hp, _ = pw.select_ics(vif_frames(D_hp["ics"]), comp_hp, 10, "kept")
var_rows, var_cv = [], {}
for name, kw in [
        ("R2: corrected signal kept high-passed", dict(ics=D_hp["ics"], keep=keep_hp)),
        ("R3 b: keypoint width as pupil", dict(ds_kw={"pupil_key": "pupil_kp"})),
        ("R6: ICs selected with the centred VIF", dict(keep=keep10c)),
        ("R7 b: folds of recordings (pooled over mice)", dict(cv_kw={"fold_unit": "recording"})),
        ("R8 b: null shuffles validation labels too", dict(cv_kw={"null_scope": "both"}))]:
    o, cv = variant(name, **kw)
    var_rows.append(o)
    var_cv[name] = cv
vt = pd.DataFrame(var_rows)
print(vt[["reading", "samples", "pos", "val_auc_model", "val_auc_null", "val_auc_p", "s"]].round(4).to_string(index=False))
print("Folds with an AUC (R7 b; a fold whose validation recordings hold no positive label has none):",
      int(np.isfinite(var_cv["R7 b: folds of recordings (pooled over mice)"].model_val_auc).sum()), "of 5")

In [ ]:
# Every combination of the cheap readings under R5 b, at the claim's seed: R2 (a, or the corrected signal
# kept high-passed), R3 (a, b), R7 (a, b), R8 (a, b), with R4 a and R6 (VIF of the paper's code).
import itertools
grid_rows, grid_cv = [], {}
t0 = time.time()
for r2, r3, r7, r8 in itertools.product(("a", "hp"), ("a", "b"), ("a", "b"), ("a", "b")):
    ics, keep = (D["ics"], keep10) if r2 == "a" else (D_hp["ics"], keep_hp)
    ds = pw.make_dataset(sessions, ics, keep, cutoff=0.1, selection="rule", pupil_key="pupil" if r3 == "a" else "pupil_kp")
    cv = pw.run_cv(ds, SEED1, fold_unit="epoch" if r7 == "a" else "recording", null_scope="train" if r8 == "a" else "both")
    grid_cv[(r2, r3, r7, r8)] = cv
    row = dict(R2=r2, R3=r3, R7=r7, R8=r8, samples=len(ds["y"]), epochs=len(np.unique(ds["unit"])))
    for m in ("val_auc", "val_loss", "val_acc"):
        r = pw.paired(cv, m)
        row[m + "_model"], row[m + "_null"], row[m + "_p"] = r["model"], r["null"], r["p"]
    row["folds"] = pw.paired(cv, "val_auc")["n"]
    grid_rows.append(row)
gt = pd.DataFrame(grid_rows)
print(gt[["R2", "R3", "R7", "R8", "samples", "epochs", "folds", "val_auc_model", "val_auc_null", "val_auc_p"]].round(4).to_string(index=False))
sig = gt[(gt.val_auc_p < 0.05) & (gt.val_auc_model > gt.val_auc_null)]
print(f"\nCombinations with the model significantly above null: {len(sig)} of {len(gt)} "
      f"(about {0.025 * len(gt):.1f} expected by chance at p < 0.05 one-sided); model AUC within 0.82 +/- 0.082 in any: "
      f"{bool((abs(sig.val_auc_model - 0.82) <= 0.082).any())}; ({time.time() - t0:.0f} s)")

In [ ]:
# Every combination above null at the claim's seed, under the four other seeds, and at the six cutoffs
# (rest-needs-0-1-hz asks for the cutoffs in a combination where the 0.1 Hz decoder is above null).
SEED3 = pw.seed_of("rest-needs-0-1-hz")
def grid_ds(key, cutoff=0.1):
    r2, r3, r7, r8 = key
    ics, keep = (D["ics"], keep10) if r2 == "a" else (D_hp["ics"], keep_hp)
    return pw.make_dataset(sessions, ics, keep, cutoff=cutoff, selection="rule", pupil_key="pupil" if r3 == "a" else "pupil_kp")
def grid_kw(key):
    return dict(fold_unit="epoch" if key[2] == "a" else "recording", null_scope="train" if key[3] == "a" else "both")
t0 = time.time()
sig_keys = [tuple(r) for r in sig[["R2", "R3", "R7", "R8"]].itertuples(index=False)]
gs_rows, gc_rows = [], []
for key in sig_keys:
    for off in (0, 1000, 2000, 3000, 4000):
        cv = grid_cv[key] if off == 0 else pw.run_cv(grid_ds(key), SEED1 + off, **grid_kw(key))
        r = pw.paired(cv, "val_auc")
        gs_rows.append(dict(combo=" ".join(key), seed=f"claim+{off}", model_auc=r["model"], null_auc=r["null"], p=r["p"],
                            above=bool(r["p"] < 0.05 and r["model"] > r["null"]), folds=r["n"]))
    for c in (0.05, 0.1, 0.2, 0.5, 1.0, 2.0):
        cv = pw.run_cv(grid_ds(key, c), SEED3, **grid_kw(key))
        r = pw.paired(cv, "val_auc")
        gc_rows.append(dict(combo=" ".join(key), cutoff_hz=c, model_auc=r["model"], null_auc=r["null"], p=r["p"],
                            above_null=bool(r["p"] < 0.05 and r["model"] > r["null"]), folds=r["n"],
                            aucs=np.round(cv.model_val_auc, 3).tolist()))
if sig_keys:
    print("Combinations above null at the claim's seed (R2 R3 R7 R8), under five seeds:")
    print(pd.DataFrame(gs_rows).round(4).to_string(index=False))
    print("\nThe same combinations at the six cutoffs (seed of rest-needs-0-1-hz):")
    gct = pd.DataFrame(gc_rows)
    print(gct.round(4).to_string(index=False))
    for key in sig_keys:
        g = gct[gct.combo == " ".join(key)].set_index("cutoff_hz")
        ref = np.array(g.loc[0.1, "aucs"], float)
        others = [0.05, 0.2, 0.5, 1.0, 2.0]
        ps = [stats.ranksums(ref[np.isfinite(ref)], np.array(g.loc[c, "aucs"], float)[np.isfinite(np.array(g.loc[c, "aucs"], float))]).pvalue for c in others]
        print(" ".join(key), ": Wilcoxon rank-sum, 0.1 Hz against", dict(zip(others, np.round(multipletests(ps, method="holm")[1], 4))), "(Holm)")
else:
    print("No combination is above null at the claim's seed.")
print(f"({time.time() - t0:.0f} s)")

**Result: partially reproduced (resting on assumptions; seed sensitive).** Under the main
combination with R5 a the models reach a mean validation AUC of 0.625 against 0.648 for the null
models (paired t = -1.32, p = 0.259), and with R5 b 0.621 against 0.550 (p = 0.430); the expected
0.82 +/- 0.082 is not reached. Over the five seeds R5 a is never above null (p = 0.26 to 0.87, the
model lower at three seeds); R5 b is above null at one seed of five (claim + 2000, 0.569 against
0.460, p = 0.011; the others p = 0.37 to 0.94). The five other readings run one at a time under R5 a
are not significant (p = 0.38 to 0.90). Of the 16 combinations of R2, R3, R7 and R8 under R5 b, 2
have the model significantly above null at the claim's seed (about 0.4 expected by chance): the
measured-signal correction with the keypoint pupil, folds of recordings and the validation labels
shuffled too (R2 a, R3 b, R7 b, R8 b; 0.696 against 0.514, p = 0.0014), and the high-passed
correction with the deposited pupil, folds of recordings and the validation labels shuffled (R2 hp,
R3 a, R7 b, R8 b; 0.614 against 0.482, p = 0.039). Under five seeds the first is above null at four
(p = 0.0014 to 0.039; at claim + 2000 0.587 against 0.551, p = 0.59) and the second at two (the
claim's seed and claim + 4000; the others p = 0.14 to 0.91). In both, the null models are scored on
shuffled validation labels (R8 b), which puts their AUC near 0.5 (0.45 to 0.57). Every mean model
AUC in the two notebooks is below 0.738, the lower end of the tolerance (the highest is 0.722, the
main combination at 0.2 Hz under R5 a), so the effect is weaker than the paper's: partially
reproduced, and seed sensitive. The every-frame runs (R4 b) in the features notebook are not above
null at any seed. Under R5 a the within-epoch shuffle changes only 12.8% of the training labels, as
most epoch windows are all negative or all positive, so the null model is close to the model by
construction; under R5 b it changes 48%. The models learn little: their training AUC is 0.684 under
R5 a and 0.561 under R5 b (the paper's example model 0.928), and the epochs of lowest validation
loss are 1, 4, 2, 1 and 4 under R5 a and 10, 2, 1, 1 and 10 under R5 b.

## rest-loss-accuracy-vs-null

> The resting-state decoder has lower validation loss and higher validation accuracy than the null
> models. (Expected: loss 0.54 against 0.68, accuracy 0.75 against 0.58; paired t-tests p < 0.05.)

Same models as rest-decoding-above-null (its seed, as this claim depends on it); accuracy at a
threshold of 0.5 at the epoch of lowest validation loss.

In [ ]:
for name, cv in (("R5 a", cv_main), ("R5 b", cv_rule)):
    for m, better in (("val_loss", "lower"), ("val_acc", "higher")):
        r = pw.paired(cv, m)
        ok = (r["model"] < r["null"]) if better == "lower" else (r["model"] > r["null"])
        print(f"{name}, {m}: model {r['model']:.3f}, null {r['null']:.3f}, t = {r['t']:.2f}, p = {r['p']:.3g}, "
              f"model {better} than null: {ok}")
    print(f"{name}: share of negative labels in the validation folds (accuracy of always saying 'no dilation'):",
          np.round(1 - cv.pos_val, 3).tolist())
print("Expected: loss 0.54 +/- 0.054 (model), 0.68 +/- 0.068 (null); accuracy 0.75 +/- 0.075, 0.58 +/- 0.058")
rows_s = []
for (combo, off), cv in sorted(cv_by_seed.items()):
    a, b = pw.paired(cv, "val_loss"), pw.paired(cv, "val_acc")
    rows_s.append(dict(combo=combo, seed=f"claim+{off}", loss_model=a["model"], loss_null=a["null"], loss_p=a["p"],
                       acc_model=b["model"], acc_null=b["null"], acc_p=b["p"]))
print(pd.DataFrame(rows_s).round(4).to_string(index=False))
print("\nOther readings under R5 a (claim's seed):")
print(vt[["reading", "val_loss_model", "val_loss_null", "val_loss_p", "val_acc_model", "val_acc_null", "val_acc_p"]].round(4).to_string(index=False))
both = gt[(gt.val_loss_p < 0.05) & (gt.val_loss_model < gt.val_loss_null) & (gt.val_acc_p < 0.05) & (gt.val_acc_model > gt.val_acc_null)]
print("\nCombinations under R5 b with both loss lower and accuracy higher than null at p < 0.05:", len(both), "of", len(gt))
print(gt[["R2", "R3", "R7", "R8", "val_loss_model", "val_loss_null", "val_loss_p", "val_acc_model", "val_acc_null", "val_acc_p"]].round(4).to_string(index=False))

**Result: not reproduced (resting on assumptions).** Under R5 a the validation loss is 0.667 against
0.650 for the null models (t = 2.72, p = 0.053, the model's loss the higher) and the accuracy 0.559
against 0.575 (p = 0.748); under R5 b the loss is 0.686 against 0.688 (p = 0.645) and the accuracy
0.583 against 0.555 (p = 0.374). The paper's loss 0.54 and accuracy 0.75 are not reached. No seed
gives either test significant (loss p = 0.053 to 0.96, accuracy p = 0.23 to 0.98; under R5 b at
claim + 1000 and claim + 4000 the model's and the null model's accuracies are equal, so no p is
given). The five readings under R5 a do not either: the smallest p is 0.050, the accuracy with the
keypoint pupil (R3 b), where the model's accuracy is the lower. Of the 16 combinations under R5 b,
none has both; one has the loss significantly lower (R2 a, R3 b, R7 b, R8 b: 0.675 against 0.687, p
= 0.0091) with the accuracy not significantly higher (0.573 against 0.556, p = 0.187). Under the
every-frame reading (R4 b, features notebook) the loss does not differ (p = 0.131 and 0.171) and
model and null have the same accuracy, the share of negative samples.

## rest-needs-0-1-hz

> Spontaneous pupil dilation could not be predicted above chance from cortical activity low-pass
> filtered at 2 Hz; a 0.1 Hz low-pass filter was required for significant decoding. The 0.1 Hz AUC
> did not differ significantly from that at any other cutoff tested (0.05, 0.2, 0.5, 1 and 2 Hz).
> (Expected AUC 0.86 +/- 0.086 at 0.1 Hz and 0.51 +/- 0.051 at 2 Hz.)

The criterion asks for the cutoffs in the reading combination that reproduces
rest-decoding-above-null, if one does, and otherwise in all of them. They are run in the main
combination under R5 a and under R5 b, at this claim's seed and four others (below), and, at this
claim's seed, in each combination above null at rest-decoding-above-null's seed (the cell after the
grid of combinations above; the R4 b runs in the features notebook would be too, but none is above
null). The ICs are those selected at 0.1 Hz, the paper's resting setting, for every cutoff
[assumption: the paper does not say whether ICs were reselected per cutoff].

In [ ]:
SEED3 = pw.seed_of("rest-needs-0-1-hz")
cut_rows, cut_cv = [], {}
for combo, kw in (("main", {}), ("R5 b", {"selection": "rule"})):
    for c in (0.05, 0.1, 0.2, 0.5, 1.0, 2.0):
        ds = pw.make_dataset(sessions, D["ics"], keep10, cutoff=c, **kw)
        cv = pw.run_cv(ds, SEED3)
        cut_cv[(combo, c)] = cv
        r = pw.paired(cv, "val_auc")
        cut_rows.append(dict(combo=combo, cutoff_hz=c, model_auc=r["model"], null_auc=r["null"], t=r["t"], p=r["p"],
                             above_null=bool(r["p"] < 0.05 and r["model"] > r["null"]),
                             median_auc=float(cv.model_val_auc.median())))
ct = pd.DataFrame(cut_rows)
print(ct.round(4).to_string(index=False))
for combo in ("main", "R5 b"):
    ref = cut_cv[(combo, 0.1)].model_val_auc
    others = [c for c in (0.05, 0.2, 0.5, 1.0, 2.0)]
    ps = [stats.ranksums(ref, cut_cv[(combo, c)].model_val_auc).pvalue for c in others]
    ph = multipletests(ps, method="holm")[1]
    print(f"{combo}: Wilcoxon rank-sum, 0.1 Hz against", dict(zip(others, np.round(ph, 4))), "(Holm-corrected)")

In [ ]:
rows_s = []
for combo, kw in (("main", {}), ("R5 b", {"selection": "rule"})):
    for off in (0, 1000, 2000, 3000, 4000):
        for c in (0.1, 2.0):
            cv = cut_cv[(combo, c)] if off == 0 else pw.run_cv(pw.make_dataset(sessions, D["ics"], keep10, cutoff=c, **kw), SEED3 + off)
            r = pw.paired(cv, "val_auc")
            rows_s.append(dict(combo=combo, seed=f"claim+{off}", cutoff_hz=c, model_auc=r["model"], null_auc=r["null"], p=r["p"],
                               above_null=bool(r["p"] < 0.05 and r["model"] > r["null"])))
print(pd.DataFrame(rows_s).round(4).to_string(index=False))
print(f"\nelapsed {time.time() - started:.0f} s")

**Result: not reproduced (resting on assumptions).** No combination reproduces
rest-decoding-above-null (none reaches its tolerance), so the criterion is judged in every
combination run at the cutoffs. In the main combination the 0.1 Hz models are not above their null
models, under R5 a (0.661 against 0.629, p = 0.085) or R5 b (0.555 against 0.459, p = 0.381), at any
of the five seeds (R5 a p = 0.085 to 0.94; R5 b p = 0.15 to 0.84). The part of absence holds there:
at 2 Hz the models are not above null (R5 a 0.607 against 0.616, p = 0.872; R5 b 0.533 against
0.555, p = 0.725; no seed significant), and no cutoff is above null in either (p >= 0.085). In the
two combinations above null at rest-decoding-above-null's seed, run here at this claim's seed, the
result splits. With R2 a, R3 b, R7 b, R8 b the claim's pattern holds: 0.1 Hz is above null (0.718
against 0.504, p = 0.009), 2 Hz is not (0.534 against 0.492, p = 0.61), and no other cutoff is (p =
0.10 to 0.55). With R2 hp, R3 a, R7 b, R8 b it does not: 0.1 Hz is not above null (0.479 against
0.475, p = 0.97), while 0.2, 0.5 and 1 Hz are (0.713, 0.670 and 0.639; p = 0.005, 0.005 and 0.039)
and 2 Hz is not (0.610 against 0.511, p = 0.13). The claim holds in one of the four combinations run
at the cutoffs and fails in three, where the 0.1 Hz model is not above null. The 2 Hz AUC of the
main combination (0.607 under R5 a) is far from the paper's 0.51. As in the paper, the
Holm-corrected rank-sum tests of 0.1 Hz against every other cutoff are not significant in any
combination (p >= 0.14).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| rest-decoding-above-null | partially reproduced (assumed; seed sensitive; above null in two of 16 combinations under R5 b, one of them at 4 of 5 seeds; mean AUC at most 0.72) |
| rest-loss-accuracy-vs-null | not reproduced (assumed) |
| rest-needs-0-1-hz | not reproduced (assumed; the claim's pattern holds in 1 of the 4 combinations run at the cutoffs) |
| rest-ic-count-and-window | not reproduced (assumed; 10 ICs highest only in a tie with the same 7-IC set, 1.5 s never) |
| rest-vif-below-10 | not reproduced (assumed) |
| rest-shap-medial-anterior-first | not reproduced (assumed; no kept IC matches the medial anterior description) |
| 18 claims on the paper's stimulation recordings | blocked: data missing |

**How the analysis differs from the paper's.** The paper does not say which three mice or which
resting sessions it used; the first three mice recorded, with their first three resting sessions,
are used, and all 25 mice (R1 b) were not run (about 90 sessions to stream, some three hours). The
ICs it kept "by visual inspection" are replaced by a repeatable rule (the most compact maps), then
pruned with the VIF as the paper's code computes it (no intercept); with that VIF only 7 ICs pass
where the paper keeps 10. Frames are binned 2 x 2 (144 x 144) and the decomposition is made on a
common atlas grid; the SVD keeps 306 components, the rank of the corrected data, in place of 400.
The SVD temporal components keep their singular values, as in the paper's code: divided by them,
as the Methods write, they have equal variance, and the 30 dimensions jadeR keeps out of 200 are
then chosen by rounding, so that two machines gave two different sets of ICs. JADE runs to
convergence (an earlier cap of 100 sweeps stopped it short), and the numerical kernels are pinned. The decomposition is then the same on every machine tried,
but the GRU fits are not always: the final run, on an AMD Zen 5 processor, gave some fits that differ
from those of the round-2 run, and the numbers here are the final run's. The GRU is written in PyTorch with Keras'
defaults, and SHAP is computed by integrated gradients from the mean input in place of
DeepExplainer. Failed pupil detections are replaced. The epoch selection is run under both
readings (none, and the stimulus data's rule); the other readings are run one at a time under the
first and in every combination of R2, R3, R7 and R8 under the second, with R4 b under the second
only.

**What was found about the dataset.** The files hold everything the resting analysis needs: frames
of both channels, pupil diameter and keypoints on the imaging clock, atlas ROIs and the atlas
transform. The excitation wavelengths of the two imaging planes are swapped in the metadata. The
deposited pupil diameter holds unflagged failed detections (values up to 2.9e21 pixels) in two of
the nine sessions read. Six of the 98 resting files hold no behaviour and so no pupil.